In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

#### 1) Import the respective csv file: 'solar-auctions.csv'

In [ ]:
lcoe_solar = pd.read_csv('solar-auctions.csv', encoding='ISO-8859-1')

In [ ]:
# Keep only contracts that still look active / economically relevant
valid_statuses = [
    'Sem Pendências',
    'Data de Suprimento Alterada',
    'Antecipação de Contrato',
    'Atraso'  # optional: keep if you still want delayed-but-active projects
]

lcoe_solar = lcoe_solar[lcoe_solar['status'].isin(valid_statuses)].copy()

In [ ]:
lcoe_solar.rename(columns={'invest_auction': 'capex'}, inplace=True)

In [ ]:
lcoe_solar.head()

#### 2) Create an 'auction_year' variable

In [ ]:
lcoe_solar['date'] = pd.to_datetime(lcoe_solar['date'], dayfirst=True)
lcoe_solar['year'] = lcoe_solar['date'].dt.year
lcoe_solar['month'] = lcoe_solar['date'].dt.month

#### 3) Obtain Capacity factor and Expected Energy Produced in One Year

In [ ]:
lcoe_solar['capacity_factor'] = lcoe_solar['physical_guarantee_mwave'] / lcoe_solar['nominal_capacity_mw']

In [ ]:
lcoe_solar['exp_energy_prod'] = lcoe_solar['capacity_factor'] * lcoe_solar['nominal_capacity_mw'] * 8760

#### 4) Obtain OPEX (1.45% assumption) and OPEX adjusted for inflation assuming 25 years of lifetime 

In [ ]:
opex_percentage = 0.01449  # 1.449% derived from the EPE report
lifetime = 25 #assuming 25 years of lifetime
lcoe_solar['opex_yearly'] = (lcoe_solar['capex'] * opex_percentage).round(2)

In [ ]:
#calculate_opex_inflated function calculates the yearly OPEX, adjusted for inflation and scaled by the nominal capacity, for each of the 25 years.
def calculate_opex_inflated(opex_yearly, inflation_rate, lifetime=25):
    opex_inflated = [(opex_yearly * ((1 + inflation_rate) ** t)) for t in range(1, lifetime + 1)]
    return opex_inflated

In [ ]:
#Inflation adjustment to each project's annual OPEX. The 'lambda' function takes each row of the DataFrame as input (row) and passes the necessary variables to the calculate_opex_inflated function.
lcoe_solar['opex_inflated'] = lcoe_solar.apply(lambda row: calculate_opex_inflated(row['opex_yearly'], 0.035), axis=1)

In [ ]:
lcoe_solar[['power_plant_name', 'opex_yearly', 'opex_inflated']].head()

#### 5) Calculate both OPEX and CAPEX Baseline

In [ ]:
def calculate_opex_baseline(row):
    total_energy_prod = row['exp_energy_prod'] * lifetime  # Total energy production over 25 years
    total_opex_inflated = sum(row['opex_inflated'])  # Sum of all inflated OPEX values over the years
    opex_baseline = total_opex_inflated / total_energy_prod  # OPEX baseline contribution to LCOE
    return opex_baseline

In [ ]:
lcoe_solar['opex_baseline'] = lcoe_solar.apply(calculate_opex_baseline, axis=1)

In [ ]:
def calculate_capex_baseline(row):
    total_energy_prod = row['exp_energy_prod'] * lifetime
    capex_baseline = row['capex'] / total_energy_prod
    return capex_baseline

In [ ]:
lcoe_solar['capex_baseline'] = lcoe_solar.apply(calculate_capex_baseline, axis=1)

#### 6) Calculate the Total LCOE Baseline (0% WACC)

In [ ]:
lcoe_solar['lcoe_baseline'] = lcoe_solar['capex_baseline'] + lcoe_solar['opex_baseline']
lcoe_solar[['power_plant_name', 'capex_baseline', 'opex_baseline', 'lcoe_baseline']].head()

#### 7) Add the WACC: Harmonize 'bndes_wacc_calculations.csv' and merge with auction dataset

In [ ]:
wacc_data = pd.read_csv('bndes_wacc_calculations.csv', encoding='ISO-8859-1')

In [ ]:
wacc_data['date'] = pd.to_datetime(wacc_data['date'], dayfirst=True, errors='coerce')

In [ ]:
wacc_data['year'] = wacc_data['date'].dt.year
wacc_data['month'] = wacc_data['date'].dt.month

In [ ]:
wacc_data

In [ ]:
lcoe_solar = pd.merge(lcoe_solar, wacc_data[['year', 'month', 'cost_of_capital']], on=['year', 'month'], how='left')

In [ ]:
lcoe_solar[['power_plant_name', 'date', 'cost_of_capital']].head()

#### 8) Calculate Total LCOE with WACC

In [ ]:
#Discounted OPEX using the respective WACC
def calculate_discounted_opex(row, lifetime=25):
    wacc = row['cost_of_capital'] / 100  # Convert WACC percentage to a decimal
    discounted_opex = [row['opex_inflated'][t-1] / ((1 + wacc) ** t) for t in range(1, lifetime + 1)]
    total_discounted_opex = sum(discounted_opex)
    return total_discounted_opex

In [ ]:
lcoe_solar['discounted_opex'] = lcoe_solar.apply(calculate_discounted_opex, axis=1)

In [ ]:
#Discounted Energyg Production using the respective WACC
def calculate_discounted_energy(row, lifetime=25):
    wacc = row['cost_of_capital'] / 100  # Convert WACC percentage to a decimal
    annual_energy = row['exp_energy_prod']  # Annual expected energy production
    discounted_energy = [annual_energy / ((1 + wacc) ** t) for t in range(1, lifetime + 1)]
    total_discounted_energy = sum(discounted_energy)
    return total_discounted_energy

In [ ]:
lcoe_solar['discounted_energy'] = lcoe_solar.apply(calculate_discounted_energy, axis=1)

In [ ]:
#Calculate the CAPEX accounting for WACC by dividing WACC by the discounted energy production
lcoe_solar['capex_wacc'] = lcoe_solar['capex'] / lcoe_solar['discounted_energy']

In [ ]:
#Same but with OPEX
lcoe_solar['opex_wacc'] = lcoe_solar['discounted_opex'] / lcoe_solar['discounted_energy']

In [ ]:
lcoe_solar['lcoe_wacc'] = lcoe_solar['capex_wacc'] + lcoe_solar['opex_wacc']

In [ ]:
lcoe_solar[['power_plant_name', 'capex_wacc', 'opex_wacc', 'lcoe_wacc']].head()

#### 9) Calculate the change in Financing Costs from 2014 to 2022 (Δi)

In [ ]:
lcoe_wacc_2014 = lcoe_solar[lcoe_solar['year'] == 2014]['lcoe_wacc'].mean()
lcoe_baseline_2014 = lcoe_solar[lcoe_solar['year'] == 2014]['lcoe_baseline'].mean()
delta_2014 = lcoe_wacc_2014 - lcoe_baseline_2014
print(delta_2014)

In [ ]:
lcoe_wacc_2022 = lcoe_solar[lcoe_solar['year'] == 2022]['lcoe_wacc'].mean()
lcoe_baseline_2022 = lcoe_solar[lcoe_solar['year'] == 2022]['lcoe_baseline'].mean()
delta_2022 = lcoe_wacc_2022 - lcoe_baseline_2022
print(delta_2022)

In [ ]:
delta_i = delta_2022 - delta_2014

In [ ]:
print(f"Change in financing costs from 2014 to 2022: {delta_i.round(2)} BRL/MWh.")

### 10) Data Visualization 

In [ ]:
import plotly.express as px

##### Nominal Capacity per Auction Year

In [ ]:
nominal_capacity_per_year = lcoe_solar.groupby('year')['nominal_capacity_mw'].sum()

In [ ]:
# Assuming 'nominal_capacity_per_year' is already calculated
fig = px.bar(nominal_capacity_per_year.reset_index(),
             x='year', 
             y='nominal_capacity_mw',
             title='Total Nominal Capacity Auctioned by Year',
             labels={'year': 'Auction Year', 'nominal_capacity_mw': 'Total Nominal Capacity (MW)'},
             color_discrete_sequence=['#1f77b4'])

# Update layout to add grid lines similar to what plt.grid does
fig.update_layout(
    xaxis_title='Auction Year',
    yaxis_title='Total Nominal Capacity (MW)',
    plot_bgcolor='white',
    xaxis=dict(showgrid=False),
    yaxis=dict(showgrid=True, gridcolor='LightGray')
)

fig.show()

##### Currency exchange BRL/USD and 2022 inflation adjustment

In [ ]:
currency_file_path = 'currency-brl-usd.csv'
currency_data = pd.read_csv(currency_file_path)

In [ ]:
currency_data['year'] = pd.to_datetime(currency_data['year'], format='%Y').dt.year

In [ ]:
lcoe_solar = pd.merge(lcoe_solar, currency_data[['year', 'brl_usd']], on='year', how='left')

In [ ]:
# New CAPEX per MW variable
lcoe_solar['capex_mw'] = lcoe_solar['capex'] / lcoe_solar['nominal_capacity_mw']

In [ ]:
# Converting CAPEX per MW to nominal USD using the exchange rate
lcoe_solar['capex_mw_usd'] = lcoe_solar['capex_mw'] / lcoe_solar['brl_usd']

In [ ]:
us_cpi = pd.read_csv('us-cpi.csv')
us_cpi['date'] = pd.to_datetime(us_cpi['date'], errors='coerce')

In [ ]:
us_cpi['date'] = pd.to_datetime(us_cpi['date'], format='%d/%m/%y')
us_cpi['year'] = us_cpi['date'].dt.year
us_cpi['month'] = us_cpi['date'].dt.month

In [ ]:
# Merge CPI data based on 'year' and 'month'
lcoe_solar = pd.merge(lcoe_solar, us_cpi, left_on=['year', 'month'], right_on=['year', 'month'], how='left')

# Locate the value of December 2022 CPI for adjustment
cpi_dec_2022 = us_cpi[(us_cpi['year'] == 2022) & (us_cpi['month'] == 12)]['cpi'].values[0]

# Adjust nominal USD for inflation to 2022 USD
lcoe_solar['capex_mw_usd_2022'] = lcoe_solar['capex_mw_usd'] * (cpi_dec_2022 / lcoe_solar['cpi'])

##### Auctions visualization

##### Capacity Factor

In [ ]:
import plotly.graph_objects as go

In [ ]:
fig = px.scatter(lcoe_solar, x='year', y='capacity_factor',
                 title='Capacity Factor per Project Over the Years',
                 labels={'year': 'Auction Year', 'capacity_factor': 'Capacity Factor'},
                 opacity=0.7, color_discrete_sequence=['darkblue'])

# Linear fit for a trend line
z = np.polyfit(lcoe_solar['year'], lcoe_solar['capacity_factor'], 1)
p = np.poly1d(z)

# Add the trend line to the plot
fig.add_trace(go.Scatter(
    x=lcoe_solar['year'], 
    y=p(lcoe_solar['year']), 
    mode='lines',
    line=dict(color='red', dash='dash', width=2),
    name='Trend Line'
))

# Customize layout to add grid lines
fig.update_layout(
    xaxis_title='Auction Year',
    yaxis_title='Capacity Factor',
    plot_bgcolor='white',
    xaxis=dict(showgrid=True, gridcolor='LightGray'),
    yaxis=dict(showgrid=True, gridcolor='LightGray'),
    legend=dict(x=0.05, y=0.95)  # Positioning the legend
)

# Show the plot
fig.show()

##### 2022 USD CAPEX per MW

In [ ]:
fig = px.scatter(lcoe_solar, x='year', y='capex_mw_usd_2022',
                 title='Investment per MW by Project Over the Years',
                 labels={'year': 'Year', 'capex_mw_usd_2022': 'CAPEX per MW (USD 2022/MW)'},
                 opacity=0.7, color_discrete_sequence=['darkblue'])

# Calculate the trend line (linear fit)
z = np.polyfit(lcoe_solar['year'], lcoe_solar['capex_mw_usd_2022'], 1)
p = np.poly1d(z)

# Add the trend line to the plot
fig.add_trace(go.Scatter(
    x=lcoe_solar['year'], 
    y=p(lcoe_solar['year']), 
    mode='lines',
    line=dict(color='red', dash='dash', width=2),
    name='Trend Line'
))

# Customize layout to add grid lines
fig.update_layout(
    xaxis_title='Year',
    yaxis_title='CAPEX per MW (USD 2022/MW)',
    plot_bgcolor='white',
    xaxis=dict(showgrid=True, gridcolor='LightGray'),
    yaxis=dict(showgrid=True, gridcolor='LightGray'),
    legend=dict(x=0.05, y=0.95)  # Positioning the legend
)

# Show the plot
fig.show()

In [ ]:
irena_data = {
    'year': [2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022],
    'capex_mw_usd_2022': [2652000, 2016000, 1833000, 1586000, 1355000, 1120000, 983000, 917000, 876000]
}

irena_df = pd.DataFrame(irena_data)

In [ ]:
fig = px.scatter(lcoe_solar, x='year', y='capex_mw_usd_2022',
                 title='Investment per MW by Project Over the Years',
                 labels={'year': 'Year', 'capex_mw_usd_2022': 'CAPEX per MW (USD 2022/MW)'},
                 opacity=0.7, color_discrete_sequence=['darkblue'])

# Calculate the trend line (linear fit) for PV projects
z = np.polyfit(lcoe_solar['year'], lcoe_solar['capex_mw_usd_2022'], 1)
p = np.poly1d(z)

# Add the trend line for PV projects
fig.add_trace(go.Scatter(
    x=lcoe_solar['year'], 
    y=p(lcoe_solar['year']), 
    mode='lines',
    line=dict(color='red', dash='dash', width=2),
    name='PV Project Trend Line'
))

# Add IRENA data to the plot as a separate scatter trace
fig.add_trace(go.Scatter(
    x=irena_df['year'], 
    y=irena_df['capex_mw_usd_2022'],
    mode='markers+lines',
    marker=dict(color='orange', size=8, symbol='circle'),
    line=dict(color='orange', width=1, dash='solid'),
    name='IRENA Global Averages'
))

# Customize layout to add grid lines
fig.update_layout(
    xaxis_title='Year',
    yaxis_title='CAPEX per MW (USD 2022/MW)',
    plot_bgcolor='white',
    xaxis=dict(showgrid=True, gridcolor='LightGray'),
    yaxis=dict(showgrid=True, gridcolor='LightGray'),
    legend=dict(x=0.05, y=0.95)  # Positioning the legend
)

# Show the plot
fig.show()

##### LCOE

In [ ]:
lcoe_solar['lcoe_baseline_usd'] = lcoe_solar['lcoe_baseline'] / lcoe_solar['brl_usd']
lcoe_solar['lcoe_wacc_usd'] = lcoe_solar['lcoe_wacc'] / lcoe_solar['brl_usd']

In [ ]:
# Adjust Baseline LCOE to 2022 USD
lcoe_solar['lcoe_baseline_usd_2022'] = lcoe_solar['lcoe_baseline_usd'] * (cpi_dec_2022 / lcoe_solar['cpi'])

# Adjust WACC LCOE to 2022 USD
lcoe_solar['lcoe_wacc_usd_2022'] = lcoe_solar['lcoe_wacc_usd'] * (cpi_dec_2022 / lcoe_solar['cpi'])

# Display the updated dataframe with the adjusted LCOE values
print(lcoe_solar[['year', 'lcoe_baseline_usd', 'lcoe_baseline_usd_2022', 'lcoe_wacc_usd', 'lcoe_wacc_usd_2022']].head())

In [ ]:
lcoe_summary_usd = lcoe_solar.groupby('year').agg(
    lcoe_min=('lcoe_wacc_usd_2022', 'min'),
    lcoe_max=('lcoe_wacc_usd_2022', 'max'),
    lcoe_avg=('lcoe_wacc_usd_2022', 'mean')
).reset_index()

# IRENA LCOE Data in USD/MWh
irena_lcoe_usd = {
    'year': [2014, 2015, 2017, 2018, 2019, 2021, 2022],
    'lcoe_wacc_usd_2022': [172, 129, 89, 75, 66, 51, 49]
}
irena_lcoe_usd_df = pd.DataFrame(irena_lcoe_usd)

# Create a bar chart for the LCOE range (Min-Max)
lcoe_usd_2022 = go.Figure()

lcoe_usd_2022.add_trace(go.Bar(
    x=lcoe_summary_usd['year'], 
    y=lcoe_summary_usd['lcoe_max'] - lcoe_summary_usd['lcoe_min'], 
    base=lcoe_summary_usd['lcoe_min'], 
    marker_color='#ffc502',
    name='LCOE Range (Min-Max)',
    opacity=0.7,
    marker_line=dict(color='#ff8800', width=1.5)
))

# Add a line chart for the average LCOE
lcoe_usd_2022.add_trace(go.Scatter(
    x=lcoe_summary_usd['year'], 
    y=lcoe_summary_usd['lcoe_avg'], 
    mode='lines+markers',
    line=dict(color='#4328e7'),
    marker=dict(symbol='circle'),
    name='Average LCOE with WACC',
    marker_line=dict(color='#fcfcfc', width=1)
))

# Add IRENA data to the plot as a separate scatter trace
lcoe_usd_2022.add_trace(go.Scatter(
    x=irena_lcoe_usd_df['year'], 
    y=irena_lcoe_usd_df['lcoe_wacc_usd_2022'],
    mode='markers+lines',
    marker=dict(color='#f3124a', symbol='circle'),
    line=dict(color='#f3124a', width=2, dash='solid'),
    name='IRENA Global Averages',
    marker_line=dict(color='#fcfcfc', width=1)
))

# Update the layout
lcoe_usd_2022.update_layout(
    title='LCOE with WACC over Time (2014-2022) - USD 2022',
    xaxis_title='Year',
    yaxis_title='LCOE with WACC (2022 USD/MWh)',
    plot_bgcolor='white',
    xaxis=dict(showgrid=True, gridcolor='LightGray'),
    yaxis=dict(showgrid=True, gridcolor='LightGray'),
    legend=dict(x=0.65, y=0.95),
    barmode='overlay',
    height=700,
    yaxis_range=[0, 250]
)

lcoe_usd_2022.show()

In [ ]:
lcoe_summary_usd = lcoe_solar.groupby('year').agg(
    lcoe_min=('lcoe_wacc_usd', 'min'),
    lcoe_max=('lcoe_wacc_usd', 'max'),
    lcoe_avg=('lcoe_wacc_usd', 'mean')
).reset_index()

# Create a bar chart for the LCOE range (Min-Max)
lcoe_usd = go.Figure()

lcoe_usd.add_trace(go.Bar(
    x=lcoe_summary_usd['year'], 
    y=lcoe_summary_usd['lcoe_max'] - lcoe_summary_usd['lcoe_min'], 
    base=lcoe_summary_usd['lcoe_min'], 
    marker_color='#ffc502',
    name='LCOE Range (Min-Max)',
    opacity=0.7,
    marker_line=dict(color='#ff8800', width=1.5)
))

# Add a line chart for the average LCOE
lcoe_usd.add_trace(go.Scatter(
    x=lcoe_summary_usd['year'], 
    y=lcoe_summary_usd['lcoe_avg'], 
    mode='lines+markers',
    line=dict(color='#4328e7'),
    marker=dict(symbol='circle'),
    name='Average LCOE with WACC',
    marker_line=dict(color='#fcfcfc', width=1)
))

# Update the layout
lcoe_usd.update_layout(
    title='LCOE with WACC over Time (2014-2022) - USD',
    xaxis_title='Year',
    yaxis_title='LCOE with WACC (USD/MWh)',
    plot_bgcolor='white',
    xaxis=dict(showgrid=True, gridcolor='LightGray'),
    yaxis=dict(showgrid=True, gridcolor='LightGray'),
    legend=dict(x=0.05, y=0.95),
    barmode='overlay',
    height=800,
    yaxis_range=[0, 250]
)

lcoe_usd.show()

In [ ]:
# Group the data by year and calculate min, max, and average LCOE for BRL

lcoe_summary_brl = lcoe_solar.groupby('year').agg(
    lcoe_min=('lcoe_wacc', 'min'),
    lcoe_max=('lcoe_wacc', 'max'),
    lcoe_avg=('lcoe_wacc', 'mean')
).reset_index()

# Create a bar chart for the LCOE range (Min-Max)
lcoe_brl = go.Figure()

lcoe_brl.add_trace(go.Bar(
    x=lcoe_summary_brl['year'], 
    y=lcoe_summary_brl['lcoe_max'] - lcoe_summary_brl['lcoe_min'], 
    base=lcoe_summary_brl['lcoe_min'], 
    marker_color='#ff6283',
    name='LCOE Range (Min-Max)',
    opacity=0.7,
    marker_line=dict(color='#f3124a', width=1.5)  # Outline color and width
))

# Add a line chart for the average LCOE in BRL
lcoe_brl.add_trace(go.Scatter(
    x=lcoe_summary_brl['year'], 
    y=lcoe_summary_brl['lcoe_avg'], 
    mode='lines+markers',
    line=dict(color='#4328e7'),
    marker=dict(symbol='circle'),
    name='Average LCOE with WACC'
))

# Update the layout
lcoe_brl.update_layout(
    title='LCOE with WACC over Time (2014-2022) - BRL Nominal Values',
    xaxis_title='Year',
    yaxis_title='LCOE with WACC (BRL$/MWh)',
    plot_bgcolor='white',
    xaxis=dict(showgrid=True, gridcolor='LightGray'),
    yaxis=dict(showgrid=True, gridcolor='LightGray'),
    legend=dict(x=0.65, y=0.95),
    barmode='overlay',
    height=800
)

lcoe_brl.show()

##### Financing Costs

In [ ]:
capex_2014 = lcoe_solar[lcoe_solar['year'] == 2014]['capex_baseline'].mean()
opex_2014 = lcoe_solar[lcoe_solar['year'] == 2014]['opex_baseline'].mean()
financing_2014 = delta_2014

In [ ]:
capex_2022 = lcoe_solar[lcoe_solar['year'] == 2022]['capex_baseline'].mean()
opex_2022 = lcoe_solar[lcoe_solar['year'] == 2022]['opex_baseline'].mean()
financing_2022 = delta_2022

In [ ]:
# Define the categories and values for 2014 and 2022
categories = ['2014', '2022']
capex = [capex_2014, capex_2022]
opex = [opex_2014, opex_2022]
financing = [financing_2014, financing_2022]

In [ ]:
fig = go.Figure()

# Add CAPEX bars
fig.add_trace(go.Bar(
    x=categories, 
    y=capex, 
    name='CAPEX',
    marker_color='#072ff2',
    text=[f'{c:.2f}' for c in capex],
    textposition='auto'
))

# Add OPEX bars stacked on top of CAPEX
fig.add_trace(go.Bar(
    x=categories, 
    y=opex, 
    name='OPEX',
    marker_color='#29dae4',
    base=capex,  # Stack OPEX on top of CAPEX
    text=[f'{o:.2f}' for o in opex],
    textposition='auto'
))

# Add Financing Costs bars stacked on top of OPEX
fig.add_trace(go.Bar(
    x=categories, 
    y=financing, 
    name='Financing Costs',
    marker_color='#ff6283',
    base=[i + j for i, j in zip(capex, opex)],  # Stack Financing on top of CAPEX and OPEX
    text=[f'{f:.2f}' for f in financing],
    textposition='auto'
))

# Update the layout
fig.update_layout(
    title='LCOE Components (CAPEX, OPEX, Financing Costs) for 2014 and 2022',
    xaxis_title='Year',
    yaxis_title='LCOE (BRL/MWh)',
    barmode='stack',  # Ensure bars are stacked
    plot_bgcolor='white',
    legend=dict(x=0.05, y=0.95),
    height=600
)

# Show the plot
fig.show()

In [ ]:
lcoe_solar.to_csv('lcoe_solar_analysis.csv', index=False)

In [ ]:
import os
os.environ["BROWSER_PATH"] = r"C:\Program Files (x86)\Microsoft\Edge\Application\msedge.exe"

import plotly.io as pio
print(pio.kaleido.scope)

In [ ]:
# import kaleido

In [ ]:
lcoe_usd_2022.write_image("lcoe_usd_2022.png", format='png', scale=4)
lcoe_usd.write_image("lcoe_usd.png", format='png', scale=4)
lcoe_brl.write_image("lcoe_brl.png", format='png', scale=4)

In [ ]:
lcoe_brl.write_image("lcoe_brl.png", format='png', scale=4)